# Demo Visualizations: Forli et al. 2025 — Egyptian Fruit Bat Hippocampus

This notebook verifies the NWB conversion of Forli, Fan, Qi & Yartsev (Nature 2025) and
reproduces two key figures:
- **Figure 2A**: 3D flight trajectory clusters
- **Figure 3D**: Spike phase locking to wingbeat cycle

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import signal
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import pynapple as nap
from pynwb import read_nwb

## Load NWB File

In [ ]:
NWB_FILE_PATH = "/Users/pauladkisson/Documents/CatalystNeuro/YartsevConv/nwb_output/Dataset_3_14543_240419.nwb"
nwbfile = read_nwb(NWB_FILE_PATH)

# --- Behavior series references ---
behavior_module = nwbfile.processing["behavior"]
position_series = behavior_module["Position"]["position"]
velocity_magnitude_series = behavior_module["BehavioralTimeSeries"]["velocity_magnitude"]
flight_status_series = behavior_module["BehavioralTimeSeries"]["flight_status"]

# --- IMU series references ---
accelerometer_series = behavior_module["IMU"]["accelerometer"]
gyroscope_series = behavior_module["IMU"]["gyroscope"]

# --- Echolocation series references ---
echolocation_module = behavior_module["Echolocation"]
click_waveforms_series = echolocation_module["echolocation_click_waveforms"]
click_amplitude_series = echolocation_module["echolocation_click_amplitude"]
click_power_spectrum_series = echolocation_module["echolocation_click_power_spectrum"]

# --- Ecephys references ---
ecephys_module = nwbfile.processing["ecephys"]
lfp_probe1 = ecephys_module["LFP"]["LFP_probe1"]
swr_probe1 = ecephys_module["SWR_probe1"]
swr_probe2 = ecephys_module["SWR_probe2"]
swr_probe3 = ecephys_module["SWR_probe3"]

print("NWB file loaded successfully.")

## Shared Time Vectors and PyNapple Epoch Set

In [ ]:
# Behavior time vector (120 Hz)
number_of_behavior_samples = position_series.data.shape[0]
behavior_times = (
    position_series.starting_time
    + np.arange(number_of_behavior_samples) / position_series.rate
)

# IMU time vector (500 Hz)
number_of_imu_samples = accelerometer_series.data.shape[0]
imu_times = (
    accelerometer_series.starting_time
    + np.arange(number_of_imu_samples) / accelerometer_series.rate
)

# Flight epoch IntervalSet for PyNapple
epochs_dataframe = nwbfile.epochs.to_dataframe()
flight_interval_set = nap.IntervalSet(
    start=epochs_dataframe["start_time"].values,
    end=epochs_dataframe["stop_time"].values,
)

# Load position and flight data fully (small arrays)
position_xyz = position_series.data[:]
velocity_magnitude = velocity_magnitude_series.data[:]
flight_status = flight_status_series.data[:]

print(f"Behavior samples: {number_of_behavior_samples} ({number_of_behavior_samples / position_series.rate:.1f} s)")
print(f"IMU samples: {number_of_imu_samples} ({number_of_imu_samples / accelerometer_series.rate:.1f} s)")
print(f"Flight epochs: {len(epochs_dataframe)}")

## Cell 4 — Session Metadata

In [ ]:
units_dataframe = nwbfile.units.to_dataframe()
number_of_units = len(units_dataframe)

# Count by unit quality and probe
if "unit_quality" in units_dataframe.columns and "probe" in units_dataframe.columns:
    good_units = units_dataframe[units_dataframe["unit_quality"] == "good"]
    mua_units = units_dataframe[units_dataframe["unit_quality"] == "MUA"]
    number_of_good_units = len(good_units)
    number_of_mua_units = len(mua_units)
    good_per_probe = good_units.groupby("probe").size()
elif "unit_quality" in units_dataframe.columns:
    good_units = units_dataframe[units_dataframe["unit_quality"] == "good"]
    mua_units = units_dataframe[units_dataframe["unit_quality"] == "MUA"]
    number_of_good_units = len(good_units)
    number_of_mua_units = len(mua_units)
    good_per_probe = None
else:
    number_of_good_units = number_of_units
    number_of_mua_units = 0
    good_per_probe = None

print("=" * 55)
print("SESSION METADATA")
print("=" * 55)
print(f"Session ID        : {nwbfile.session_id}")
print(f"Session start     : {nwbfile.session_start_time}")
print(f"Subject           : {nwbfile.subject}")
print(f"Experiment desc.  : {nwbfile.experiment_description}")
print()
print(f"Total units       : {number_of_units}")
print(f"  Good units      : {number_of_good_units}")
if good_per_probe is not None:
    for probe_name, count in good_per_probe.items():
        print(f"    {probe_name}: {count}")
print(f"  MUA units       : {number_of_mua_units}")
print()
print(f"Flight epochs     : {len(epochs_dataframe)}")
print(f"Session duration  : {behavior_times[-1]:.1f} s ({behavior_times[-1]/60:.1f} min)")
print()
print(f"SWR events probe1 : {len(swr_probe1.to_dataframe())}")
print(f"SWR events probe2 : {len(swr_probe2.to_dataframe())}")
print(f"SWR events probe3 : {len(swr_probe3.to_dataframe())}")
print()
print(f"Echolocation clicks: {click_amplitude_series.data.shape[0]}")
print(f"LFP probe1 shape   : {lfp_probe1.data.shape}  (samples × channels)")
print("=" * 55)

## Cell 5 — 3D Flight Trajectories (All Epochs)

In [ ]:
color_map = plt.cm.tab20
number_of_epochs = len(epochs_dataframe)

figure = plt.figure(figsize=(12, 5))
axes_3d = figure.add_subplot(121, projection="3d")
axes_xy = figure.add_subplot(122)

for epoch_index, row in epochs_dataframe.iterrows():
    epoch_color = color_map(epoch_index / number_of_epochs)
    start_sample = int((row["start_time"] - behavior_times[0]) * position_series.rate)
    stop_sample = int((row["stop_time"] - behavior_times[0]) * position_series.rate)
    start_sample = max(0, start_sample)
    stop_sample = min(number_of_behavior_samples - 1, stop_sample)

    trajectory = position_xyz[start_sample:stop_sample]
    if len(trajectory) < 2:
        continue

    axes_3d.plot(
        trajectory[:, 0], trajectory[:, 1], trajectory[:, 2],
        color=epoch_color, linewidth=0.5, alpha=0.7
    )
    axes_xy.plot(
        trajectory[:, 0], trajectory[:, 1],
        color=epoch_color, linewidth=0.5, alpha=0.7
    )

axes_3d.set_xlabel("X (m)")
axes_3d.set_ylabel("Y (m)")
axes_3d.set_zlabel("Z (m)")
axes_3d.set_title("3D Flight Trajectories (all epochs)")
axes_3d.set_xlim([-2.9, 2.9])
axes_3d.set_ylim([-2.6, 2.6])
axes_3d.set_zlim([0, 2.3])

axes_xy.set_xlabel("X (m)")
axes_xy.set_ylabel("Y (m)")
axes_xy.set_title("Top-down View (XY)")
axes_xy.set_xlim([-2.9, 2.9])
axes_xy.set_ylim([-2.6, 2.6])
axes_xy.set_aspect("equal")

scalar_map = plt.cm.ScalarMappable(cmap=color_map, norm=mcolors.Normalize(0, number_of_epochs))
figure.colorbar(scalar_map, ax=axes_xy, label="Epoch index")
plt.tight_layout()
plt.show()

## Cell 6 — Velocity Magnitude + Flight Epoch Shading

In [ ]:
figure, axes = plt.subplots(figsize=(16, 3))

axes.plot(behavior_times, velocity_magnitude, color="steelblue", linewidth=0.4, alpha=0.8)

for _, row in epochs_dataframe.iterrows():
    axes.axvspan(row["start_time"], row["stop_time"], alpha=0.15, color="orange", linewidth=0)

axes.set_xlabel("Time (s)")
axes.set_ylabel("Velocity (m/s)")
axes.set_title("Velocity Magnitude with Flight Epochs (orange shading)")
plt.tight_layout()
plt.show()

## Cell 7 — Population Spike Raster (5-minute window)

In [ ]:
raster_start_time = behavior_times[0]
raster_stop_time = raster_start_time + 300.0  # 5 minutes

# Collect spike data
unit_spike_times_list = []
unit_firing_rates = []
unit_is_good = []

session_duration = behavior_times[-1] - behavior_times[0]

for unit_index in range(len(units_dataframe)):
    spike_times = nwbfile.units["spike_times"][unit_index]
    window_mask = (spike_times >= raster_start_time) & (spike_times <= raster_stop_time)
    unit_spike_times_list.append(spike_times[window_mask])
    unit_firing_rates.append(len(spike_times) / session_duration)
    if "unit_quality" in units_dataframe.columns:
        unit_is_good.append(units_dataframe.iloc[unit_index]["unit_quality"] == "good")
    else:
        unit_is_good.append(True)

# Sort by probe + firing rate
if "probe" in units_dataframe.columns:
    sort_keys = list(zip(units_dataframe["probe"].values, unit_firing_rates))
    sorted_indices = sorted(range(len(sort_keys)), key=lambda i: (sort_keys[i][0], sort_keys[i][1]))
else:
    sorted_indices = sorted(range(len(unit_firing_rates)), key=lambda i: unit_firing_rates[i])

figure, axes = plt.subplots(figsize=(16, 8))

for raster_row, unit_index in enumerate(sorted_indices):
    spike_times_in_window = unit_spike_times_list[unit_index]
    marker_color = "black" if unit_is_good[unit_index] else "gray"
    axes.scatter(
        spike_times_in_window,
        np.full(len(spike_times_in_window), raster_row),
        s=0.3, c=marker_color, marker="|", linewidths=0.3
    )

for _, row in epochs_dataframe.iterrows():
    if row["start_time"] <= raster_stop_time and row["stop_time"] >= raster_start_time:
        axes.axvspan(row["start_time"], row["stop_time"], alpha=0.1, color="orange", linewidth=0)

axes.set_xlim([raster_start_time, raster_stop_time])
axes.set_ylim([-1, len(units_dataframe)])
axes.set_xlabel("Time (s)")
axes.set_ylabel("Unit (sorted by probe + firing rate)")
axes.set_title("Population Spike Raster — 5-minute window (black=good, gray=MUA)")
plt.tight_layout()
plt.show()

## Cell 8 — Firing Rate Distribution

In [ ]:
good_firing_rates = [unit_firing_rates[i] for i in range(len(unit_firing_rates)) if unit_is_good[i]]
mua_firing_rates = [unit_firing_rates[i] for i in range(len(unit_firing_rates)) if not unit_is_good[i]]

figure, axes = plt.subplots(figsize=(8, 4))
bin_edges = np.linspace(0, np.percentile(unit_firing_rates, 99), 50)

axes.hist(good_firing_rates, bins=bin_edges, alpha=0.7, color="steelblue", label=f"Good units (n={len(good_firing_rates)})")
axes.hist(mua_firing_rates, bins=bin_edges, alpha=0.5, color="gray", label=f"MUA units (n={len(mua_firing_rates)})")

axes.set_xlabel("Mean firing rate (Hz)")
axes.set_ylabel("Number of units")
axes.set_title("Firing Rate Distribution")
axes.legend()
plt.tight_layout()
plt.show()

## Cell 9 — LFP Trace (10-second slice, 7 channels)

In [ ]:
lfp_target_time = behavior_times[0] + 60.0  # 60 s into session
lfp_duration = 10.0  # seconds
lfp_rate = lfp_probe1.rate

lfp_start_sample = int((lfp_target_time - lfp_probe1.starting_time) * lfp_rate)
lfp_stop_sample = lfp_start_sample + int(lfp_duration * lfp_rate)
lfp_start_sample = max(0, lfp_start_sample)
lfp_stop_sample = min(lfp_probe1.data.shape[0], lfp_stop_sample)

# Load slice and convert to microvolts
lfp_slice_int16 = lfp_probe1.data[lfp_start_sample:lfp_stop_sample, :]  # (samples, channels)
lfp_slice_volts = lfp_slice_int16.astype(np.float32) * lfp_probe1.conversion
lfp_slice_microvolts = lfp_slice_volts * 1e6

lfp_time_axis = lfp_probe1.starting_time + np.arange(lfp_start_sample, lfp_stop_sample) / lfp_rate

# Pick 7 evenly spaced channels
number_of_lfp_channels = lfp_slice_microvolts.shape[1]
channel_indices = np.linspace(0, number_of_lfp_channels - 1, 7, dtype=int)

figure, axes = plt.subplots(figsize=(14, 6))
offset_scale = 200.0  # µV between traces

for plot_index, channel_index in enumerate(channel_indices):
    trace = lfp_slice_microvolts[:, channel_index]
    axes.plot(
        lfp_time_axis,
        trace + plot_index * offset_scale,
        linewidth=0.6,
        label=f"Ch {channel_index}"
    )

axes.set_xlabel("Time (s)")
axes.set_ylabel("Voltage (µV, offset by channel)")
axes.set_title("LFP Probe 1 — 10-second trace, 7 channels")
axes.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"LFP amplitude range (first channel): {lfp_slice_microvolts[:, channel_indices[0]].min():.1f} to {lfp_slice_microvolts[:, channel_indices[0]].max():.1f} µV")

## Cell 10 — LFP Spectrogram (60-second slice)

In [ ]:
spectrogram_target_time = behavior_times[0] + 60.0
spectrogram_duration = 60.0  # seconds

spectrogram_start_sample = int((spectrogram_target_time - lfp_probe1.starting_time) * lfp_rate)
spectrogram_stop_sample = spectrogram_start_sample + int(spectrogram_duration * lfp_rate)
spectrogram_start_sample = max(0, spectrogram_start_sample)
spectrogram_stop_sample = min(lfp_probe1.data.shape[0], spectrogram_stop_sample)

# Load single channel
reference_channel_index = 15
lfp_segment_int16 = lfp_probe1.data[spectrogram_start_sample:spectrogram_stop_sample, reference_channel_index]
lfp_segment_microvolts = lfp_segment_int16.astype(np.float32) * lfp_probe1.conversion * 1e6

spectrogram_frequencies, spectrogram_times, spectrogram_power = signal.spectrogram(
    lfp_segment_microvolts,
    fs=lfp_rate,
    nperseg=int(lfp_rate),      # 1-second windows
    noverlap=int(lfp_rate * 0.5),
    scaling="density",
)

# Restrict to 0–150 Hz
frequency_mask = spectrogram_frequencies <= 150

figure, axes = plt.subplots(figsize=(14, 4))
spectrogram_absolute_times = spectrogram_target_time + spectrogram_times
axes.pcolormesh(
    spectrogram_absolute_times,
    spectrogram_frequencies[frequency_mask],
    10 * np.log10(spectrogram_power[frequency_mask] + 1e-12),
    shading="auto",
    cmap="viridis"
)
axes.set_ylabel("Frequency (Hz)")
axes.set_xlabel("Time (s)")
axes.set_title(f"LFP Spectrogram — Probe 1, channel {reference_channel_index}, 60-second window")
plt.colorbar(axes.collections[0], ax=axes, label="Power (dB/Hz)")
plt.tight_layout()
plt.show()

## Cell 11 — Sharp-Wave Ripple Events

In [ ]:
swr_dataframe = swr_probe1.to_dataframe()

# Compute inter-event intervals
inter_event_intervals = np.diff(swr_dataframe["peak_time"].values)

figure = plt.figure(figsize=(14, 8))
grid_spec = gridspec.GridSpec(2, 3, figure=figure)

# Amplitude histogram
axes_amplitude = figure.add_subplot(grid_spec[0, 0])
axes_amplitude.hist(swr_dataframe["amplitude_zscore"], bins=50, color="steelblue", edgecolor="none")
axes_amplitude.set_xlabel("Amplitude (z-score)")
axes_amplitude.set_ylabel("Count")
axes_amplitude.set_title("SWR Amplitude Distribution")

# Inter-event interval histogram
axes_iei = figure.add_subplot(grid_spec[0, 1])
iei_clipped = inter_event_intervals[inter_event_intervals < 10.0]  # clip at 10 s for display
axes_iei.hist(iei_clipped, bins=50, color="coral", edgecolor="none")
axes_iei.set_xlabel("Inter-event interval (s)")
axes_iei.set_ylabel("Count")
axes_iei.set_title("SWR Inter-event Intervals")

# Duration histogram
axes_duration = figure.add_subplot(grid_spec[0, 2])
swr_durations_ms = (swr_dataframe["stop_time"] - swr_dataframe["start_time"]) * 1000
axes_duration.hist(swr_durations_ms, bins=50, color="mediumseagreen", edgecolor="none")
axes_duration.set_xlabel("Duration (ms)")
axes_duration.set_ylabel("Count")
axes_duration.set_title("SWR Duration Distribution")

# 6 example ripple traces around peak_time
# Use a fixed LFP channel for all snippets
snippet_channel_index = 15
snippet_half_duration = 0.1  # 100 ms on each side
snippet_samples = int(snippet_half_duration * lfp_rate)

# Pick 6 high-amplitude events that have valid LFP coverage
top_amplitude_events = swr_dataframe.nlargest(30, "amplitude_zscore")

example_axes_list = [figure.add_subplot(grid_spec[1, col]) for col in range(3)]
example_axes_list += [figure.add_subplot(grid_spec[1, col]) for col in range(3)]  # placeholder — replace with 6 subplots below

# Rebuild as 2x3 with the ripple traces taking the bottom 2 rows
plt.close(figure)
figure = plt.figure(figsize=(14, 8))
grid_spec = gridspec.GridSpec(3, 3, figure=figure, hspace=0.45, wspace=0.35)

axes_amplitude = figure.add_subplot(grid_spec[0, 0])
axes_amplitude.hist(swr_dataframe["amplitude_zscore"], bins=50, color="steelblue", edgecolor="none")
axes_amplitude.set_xlabel("Amplitude (z-score)")
axes_amplitude.set_ylabel("Count")
axes_amplitude.set_title("SWR Amplitude")

axes_iei = figure.add_subplot(grid_spec[0, 1])
axes_iei.hist(iei_clipped, bins=50, color="coral", edgecolor="none")
axes_iei.set_xlabel("IEI (s)")
axes_iei.set_ylabel("Count")
axes_iei.set_title("Inter-event Interval")

axes_duration = figure.add_subplot(grid_spec[0, 2])
axes_duration.hist(swr_durations_ms, bins=50, color="mediumseagreen", edgecolor="none")
axes_duration.set_xlabel("Duration (ms)")
axes_duration.set_ylabel("Count")
axes_duration.set_title("SWR Duration")

number_of_examples_plotted = 0
for _, swr_row in top_amplitude_events.iterrows():
    if number_of_examples_plotted >= 6:
        break
    peak_time = float(swr_row["peak_time"])
    center_sample = int((peak_time - lfp_probe1.starting_time) * lfp_rate)
    start_sample = center_sample - snippet_samples
    stop_sample = center_sample + snippet_samples
    if start_sample < 0 or stop_sample >= lfp_probe1.data.shape[0]:
        continue

    snippet_int16 = lfp_probe1.data[start_sample:stop_sample, snippet_channel_index]
    snippet_microvolts = snippet_int16.astype(np.float32) * lfp_probe1.conversion * 1e6
    snippet_time_ms = (np.arange(len(snippet_microvolts)) - snippet_samples) / lfp_rate * 1000

    row_index = 1 + number_of_examples_plotted // 3
    column_index = number_of_examples_plotted % 3
    axes_snippet = figure.add_subplot(grid_spec[row_index, column_index])
    axes_snippet.plot(snippet_time_ms, snippet_microvolts, color="steelblue", linewidth=0.8)
    axes_snippet.axvline(0, color="red", linewidth=0.8, linestyle="--")
    axes_snippet.set_xlabel("Time (ms)")
    axes_snippet.set_ylabel("µV")
    axes_snippet.set_title(f"SWR z={swr_row['amplitude_zscore']:.1f}")
    number_of_examples_plotted += 1

figure.suptitle("Sharp-Wave Ripples — Probe 1", fontsize=13)
plt.show()
print(f"Plotted {number_of_examples_plotted} example ripple traces.")

## Cell 12 — IMU Overview (30-second window)

In [ ]:
imu_window_start = imu_times[0] + 60.0
imu_window_duration = 30.0  # seconds
imu_rate = accelerometer_series.rate

imu_start_sample = int((imu_window_start - accelerometer_series.starting_time) * imu_rate)
imu_stop_sample = imu_start_sample + int(imu_window_duration * imu_rate)
imu_start_sample = max(0, imu_start_sample)
imu_stop_sample = min(number_of_imu_samples, imu_stop_sample)

imu_window_times = imu_times[imu_start_sample:imu_stop_sample]
acceleration_window = accelerometer_series.data[imu_start_sample:imu_stop_sample, :]  # (samples, 3)
gyroscope_window = gyroscope_series.data[imu_start_sample:imu_stop_sample, :]  # (samples, 3)

acceleration_magnitude_window = np.linalg.norm(acceleration_window, axis=1)
gyroscope_magnitude_window = np.linalg.norm(gyroscope_window, axis=1)

figure, axes_list = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes_list[0].plot(imu_window_times, acceleration_magnitude_window, color="steelblue", linewidth=0.6)
axes_list[0].set_ylabel("|Acceleration| (g)")
axes_list[0].set_title("IMU Overview — 30-second window")

axes_list[1].plot(imu_window_times, gyroscope_magnitude_window, color="darkorange", linewidth=0.6)
axes_list[1].set_ylabel("|Gyroscope| (deg/s)")
axes_list[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## Cell 13 — Echolocation Clicks

In [ ]:
click_timestamps = click_amplitude_series.timestamps[:]
click_amplitudes = click_amplitude_series.data[:]
click_waveforms = click_waveforms_series.data[:]  # (n_clicks, n_waveform_samples)
click_power_spectra = click_power_spectrum_series.data[:]  # (n_clicks, n_freq_bins)

mean_waveform = click_waveforms.mean(axis=0)
std_waveform = click_waveforms.std(axis=0)
waveform_time_axis = np.arange(mean_waveform.shape[0]) / click_waveforms_series.rate * 1000  # ms

mean_power_spectrum = click_power_spectra.mean(axis=0)
# Frequency axis: rate/2 spans n_freq_bins bins
click_waveform_rate = click_waveforms_series.rate
number_of_freq_bins = mean_power_spectrum.shape[0]
frequency_axis = np.linspace(0, click_waveform_rate / 2, number_of_freq_bins) / 1000  # kHz

figure, axes_list = plt.subplots(1, 3, figsize=(15, 4))

# Click event scatter
axes_list[0].scatter(click_timestamps, click_amplitudes, s=1, color="steelblue", alpha=0.5)
axes_list[0].set_xlabel("Time (s)")
axes_list[0].set_ylabel("Amplitude (mV)")
axes_list[0].set_title(f"Echolocation Clicks (n={len(click_timestamps)})")

# Mean waveform ± SD
axes_list[1].fill_between(
    waveform_time_axis,
    mean_waveform - std_waveform,
    mean_waveform + std_waveform,
    alpha=0.3, color="steelblue"
)
axes_list[1].plot(waveform_time_axis, mean_waveform, color="steelblue", linewidth=1.2)
axes_list[1].set_xlabel("Time (ms)")
axes_list[1].set_ylabel("Amplitude (V)")
axes_list[1].set_title("Mean Click Waveform ± SD")

# Pre-computed power spectrum
axes_list[2].plot(frequency_axis, mean_power_spectrum, color="darkorange", linewidth=1.2)
axes_list[2].set_xlabel("Frequency (kHz)")
axes_list[2].set_ylabel("Power (a.u.)")
axes_list[2].set_title("Mean Click Power Spectrum")

plt.tight_layout()
plt.show()

## Cell 14 — Figure 2A: Flight Trajectory Clusters

Approximate reproduction using start/end position clustering (simplified from the
MATLAB `FlightClus_AF_v3` which uses Fréchet distance on full trajectories).

In [ ]:
# Extract start and end 3D position for each flight epoch
flight_start_positions = np.zeros((len(epochs_dataframe), 3))
flight_end_positions = np.zeros((len(epochs_dataframe), 3))

for epoch_index, row in epochs_dataframe.iterrows():
    start_sample = int((row["start_time"] - behavior_times[0]) * position_series.rate)
    stop_sample = int((row["stop_time"] - behavior_times[0]) * position_series.rate)
    start_sample = np.clip(start_sample, 0, number_of_behavior_samples - 1)
    stop_sample = np.clip(stop_sample, 0, number_of_behavior_samples - 1)
    flight_start_positions[epoch_index] = position_xyz[start_sample]
    flight_end_positions[epoch_index] = position_xyz[stop_sample]

# 6D feature vector: [start_x, start_y, start_z, end_x, end_y, end_z]
clustering_features = np.hstack([flight_start_positions, flight_end_positions])
clustering_features_scaled = StandardScaler().fit_transform(clustering_features)

cluster_labels = AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(
    clustering_features_scaled
)

# Feeder positions (approximate, from room calibration)
feeder_positions = np.array([[2.77, 0.82], [2.79, -0.99]])

cluster_color_map = plt.cm.tab10

figure, axes = plt.subplots(figsize=(8, 7))

for epoch_index, row in epochs_dataframe.iterrows():
    start_sample = int((row["start_time"] - behavior_times[0]) * position_series.rate)
    stop_sample = int((row["stop_time"] - behavior_times[0]) * position_series.rate)
    start_sample = np.clip(start_sample, 0, number_of_behavior_samples - 1)
    stop_sample = np.clip(stop_sample, 0, number_of_behavior_samples - 1)

    trajectory = position_xyz[start_sample:stop_sample]
    if len(trajectory) < 2:
        continue

    cluster_color = cluster_color_map(cluster_labels[epoch_index] / 5)
    axes.plot(
        trajectory[:, 0], trajectory[:, 1],
        color=cluster_color, linewidth=0.8, alpha=0.75,
        label=f"Cluster {cluster_labels[epoch_index]}" if epoch_index == np.where(cluster_labels == cluster_labels[epoch_index])[0][0] else None
    )

# Mark feeders
axes.scatter(
    feeder_positions[:, 0], feeder_positions[:, 1],
    marker="*", s=300, color="gold", edgecolors="black", linewidths=0.8,
    zorder=5, label="Feeders"
)

axes.set_xlabel("X (m)", fontsize=12)
axes.set_ylabel("Y (m)", fontsize=12)
axes.set_title("Figure 2A: Flight Trajectory Clusters (top-down view)", fontsize=13)
axes.set_xlim([-2.9, 2.9])
axes.set_ylim([-2.6, 2.6])
axes.set_aspect("equal")

# Deduplicate legend
handles, labels = axes.get_legend_handles_labels()
unique_labels = dict(zip(labels, handles))
axes.legend(unique_labels.values(), unique_labels.keys(), loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("Flights per cluster:")
for cluster_id, count in cluster_counts.items():
    print(f"  Cluster {cluster_id}: {count} flights")

## Cell 15 — Figure 3D: Spike Phase Locking to Wingbeat Cycle

Follows MATLAB demo lines 289–308:
1. Bandpass accelerometer magnitude at 7–9 Hz (wingbeat frequency)
2. Extract instantaneous phase via Hilbert transform
3. Interpolate to spike times and compute Rayleigh statistics
4. Plot top 12 phase-locked units as polar histograms

In [ ]:
# Load full accelerometer (~30 MB)
acceleration_full = accelerometer_series.data[:].astype(np.float32)  # (samples, 3)
acceleration_magnitude = np.linalg.norm(acceleration_full, axis=1)

# Bandpass filter at 7–9 Hz (wingbeat frequency) using sos for numerical stability
wingbeat_low_frequency = 7.0
wingbeat_high_frequency = 9.0
filter_order = 4
filter_coefficients = signal.butter(
    filter_order,
    [wingbeat_low_frequency, wingbeat_high_frequency],
    btype="bandpass",
    fs=float(accelerometer_series.rate),
    output="sos"
)
filtered_acceleration = signal.sosfiltfilt(filter_coefficients, acceleration_magnitude)

# Extract instantaneous wingbeat phase
wingbeat_phase = np.angle(signal.hilbert(filtered_acceleration))

# Interpolate wingbeat phase to behavior timebase
wingbeat_phase_behavior = np.interp(behavior_times, imu_times, wingbeat_phase)

print(f"Accelerometer samples processed: {len(acceleration_magnitude)}")
print(f"Wingbeat phase range: [{wingbeat_phase.min():.2f}, {wingbeat_phase.max():.2f}] rad")

In [ ]:
# Compute Rayleigh statistics for each good unit
minimum_flight_spikes = 10
imu_time_start = imu_times[0]
imu_time_stop = imu_times[-1]

# Identify good units
if "unit_quality" in units_dataframe.columns:
    good_unit_indices = units_dataframe.index[units_dataframe["unit_quality"] == "good"].tolist()
else:
    good_unit_indices = list(range(len(units_dataframe)))

rayleigh_statistics = []
mean_phases = []
spike_phase_lists = []
valid_unit_indices = []

for unit_index in good_unit_indices:
    spike_times = nwbfile.units["spike_times"][unit_index]

    # Restrict to flight epochs using PyNapple
    spike_train = nap.Ts(t=spike_times, time_support=flight_interval_set)
    flight_spike_times = spike_train.t

    # Mask to IMU time range before interpolation
    imu_range_mask = (flight_spike_times >= imu_time_start) & (flight_spike_times <= imu_time_stop)
    flight_spike_times_valid = flight_spike_times[imu_range_mask]

    if len(flight_spike_times_valid) < minimum_flight_spikes:
        rayleigh_statistics.append(0.0)
        mean_phases.append(0.0)
        spike_phase_lists.append(np.array([]))
        valid_unit_indices.append(unit_index)
        continue

    spike_phases = np.interp(flight_spike_times_valid, imu_times, wingbeat_phase)

    # Mean resultant length R
    complex_phases = np.exp(1j * spike_phases)
    mean_vector = complex_phases.mean()
    resultant_length = np.abs(mean_vector)
    mean_phase = np.angle(mean_vector)

    # Rayleigh z-statistic
    number_of_spikes = len(spike_phases)
    rayleigh_z = number_of_spikes * resultant_length ** 2

    rayleigh_statistics.append(rayleigh_z)
    mean_phases.append(mean_phase)
    spike_phase_lists.append(spike_phases)
    valid_unit_indices.append(unit_index)

rayleigh_statistics = np.array(rayleigh_statistics)
mean_phases = np.array(mean_phases)

top12_indices = np.argsort(rayleigh_statistics)[::-1][:12]

print(f"Good units analyzed: {len(good_unit_indices)}")
print(f"Top Rayleigh z-statistics: {rayleigh_statistics[top12_indices[:5]]}")

In [ ]:
# Plot Figure 3D: 3×4 polar histogram grid
number_of_phase_bins = 36  # 10° per bin
bin_edges = np.linspace(-np.pi, np.pi, number_of_phase_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = 2 * np.pi / number_of_phase_bins

figure = plt.figure(figsize=(14, 10))
figure.suptitle("Figure 3D: Spike Phase Locking to Wingbeat Cycle (Top 12 Units)", fontsize=13)

for plot_index, top_index in enumerate(top12_indices):
    unit_index = valid_unit_indices[top_index]
    phases = spike_phase_lists[top_index]
    rayleigh_z = rayleigh_statistics[top_index]
    preferred_phase = mean_phases[top_index]

    counts, _ = np.histogram(phases, bins=bin_edges)
    resultant_length_unit = np.sqrt((np.cos(phases).mean()) ** 2 + (np.sin(phases).mean()) ** 2)

    axes_polar = figure.add_subplot(3, 4, plot_index + 1, projection="polar")
    axes_polar.bar(
        bin_centers, counts,
        width=bin_width,
        color="steelblue", alpha=0.8, edgecolor="none"
    )

    # Red arrow for mean resultant vector
    max_count = counts.max() if counts.max() > 0 else 1
    arrow_length = resultant_length_unit * max_count
    axes_polar.annotate(
        "",
        xy=(preferred_phase, arrow_length),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="red", lw=2.0),
    )

    axes_polar.set_title(
        f"Unit {unit_index}\nz={rayleigh_z:.1f}, n={len(phases)}",
        fontsize=8, pad=3
    )
    axes_polar.set_xticks(np.linspace(0, 2 * np.pi, 5)[:-1])
    axes_polar.set_xticklabels(["0", "π/2", "π", "3π/2"], fontsize=7)
    axes_polar.tick_params(labelsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()